In [14]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env automatically

print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("LANGSMITH loaded:", bool(os.getenv("LANGCHAIN_API_KEY")))

OPENAI_API_KEY loaded: True
LANGSMITH loaded: True


In [15]:
import os
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/calendar"]

def get_calendar_service():
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("../credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)

        with open("token.json", "w") as f:
            f.write(creds.to_json())

    return build("calendar", "v3", credentials=creds)


service = get_calendar_service()
print("Connected to Google Calendar!")

Connected to Google Calendar!


In [16]:
print(os.getcwd())

/Users/pranavr/Desktop/Pranav/Miscellaneous/Calendar_Agent/notebooks


In [ ]:
# from langchain_core.tools import tool
# from datetime import datetime

# @tool
# def get_calendar_events(date: str) -> str:
#     """
#     Get all calendar events for a given date.
#     Always call this FIRST before scheduling anything.
#     Use this to find what time slots are already taken.

#     Args:
#         date: "today" or a date in YYYY-MM-DD format
#     """
#     if date == "today":
#         date = datetime.now().strftime("%Y-%m-%d")

#     service = get_calendar_service()

#     start = f"{date}T00:00:00Z"
#     end   = f"{date}T23:59:59Z"

#     result = service.events().list(
#         calendarId="primary",
#         timeMin=start,
#         timeMax=end,
#         singleEvents=True,
#         orderBy="startTime"
#     ).execute()

#     events = result.get("items", [])

#     if not events:
#         return f"No events on {date}. Full day is free."

#     summary = f"Events on {date}:\n"
#     for e in events:
#         start_time = e["start"].get("dateTime", "all-day")
#         end_time   = e["end"].get("dateTime", "")
#         title      = e.get("summary", "Untitled")
#         if start_time != "all-day":
#             start_time = start_time[11:16]
#             end_time   = end_time[11:16]
#             summary += f"  - {title}: {start_time} to {end_time}\n"
#         else:
#             summary += f"  - {title}: all day\n"

#     return summary

# # Test it — calls your real Google Calendar
# result = get_calendar_events.invoke({"date": "today"})
# print(result)

No events on 2026-03-30. Full day is free.


In [17]:
from langchain_core.tools import tool
from datetime import datetime
import re

@tool
def get_calendar_events(date: str) -> str:
    """
    Get all calendar events for a given date.
    Always call this FIRST before scheduling anything.
    Use this to find what time slots are already taken.

    Args:
        date: "today" or a date in YYYY-MM-DD format e.g. "2026-03-31"
    """
    try:
        # handle "today"
        if date == "today":
            date = datetime.now().strftime("%Y-%m-%d")

        # validate date format
        datetime.strptime(date, "%Y-%m-%d")

        service = get_calendar_service()

        start = f"{date}T00:00:00Z"
        end   = f"{date}T23:59:59Z"

        result = service.events().list(
            calendarId="primary",
            timeMin=start,
            timeMax=end,
            singleEvents=True,
            orderBy="startTime"
        ).execute()

        events = result.get("items", [])

        if not events:
            return f"No events on {date}. Full day is free."

        summary = f"Events on {date}:\n"
        for e in events:
            start_time = e["start"].get("dateTime", "all-day")
            end_time   = e["end"].get("dateTime", "")
            title      = e.get("summary", "Untitled")
            if start_time != "all-day":
                start_time = start_time[11:16]
                end_time   = end_time[11:16]
                summary += f"  - {title}: {start_time} to {end_time}\n"
            else:
                summary += f"  - {title}: all day\n"

        return summary

    except ValueError as e:
        return f"Error: invalid date format '{date}'. Use YYYY-MM-DD e.g. 2026-03-31."
    except Exception as e:
        return f"Error accessing calendar: {str(e)}. Please try again."


@tool
def create_calendar_event(
    title: str,
    date: str,
    start_time: str,
    end_time: str
) -> str:
    """
    Create a calendar event in Google Calendar.
    Only call this after checking get_calendar_events to confirm the slot is free.
    Never double-book an existing event.

    Args:
        title:      Name of the event e.g. "Gym", "Study session"
        date:       Date in YYYY-MM-DD format e.g. "2026-03-31"
        start_time: Start in HH:MM 24hr format e.g. "09:00"
        end_time:   End in HH:MM 24hr format e.g. "10:00"
    """
    try:
        # validate date format
        datetime.strptime(date, "%Y-%m-%d")

        # validate time format — must be HH:MM
        time_pattern = re.compile(r"^\d{2}:\d{2}$")
        if not time_pattern.match(start_time):
            return f"Error: start_time '{start_time}' is invalid. Use HH:MM format e.g. 09:00"
        if not time_pattern.match(end_time):
            return f"Error: end_time '{end_time}' is invalid. Use HH:MM format e.g. 10:00"

        # validate start is before end
        start_dt = datetime.strptime(f"{date}T{start_time}", "%Y-%m-%dT%H:%M")
        end_dt   = datetime.strptime(f"{date}T{end_time}",   "%Y-%m-%dT%H:%M")
        if start_dt >= end_dt:
            return f"Error: start_time {start_time} must be before end_time {end_time}."

        service = get_calendar_service()

        event = {
            "summary": title,
            "start": {
                "dateTime": f"{date}T{start_time}:00",
                "timeZone": "America/New_York",
            },
            "end": {
                "dateTime": f"{date}T{end_time}:00",
                "timeZone": "America/New_York",
            },
        }

        service.events().insert(
            calendarId="primary",
            body=event
        ).execute()

        return f"Created '{title}' on {date} from {start_time} to {end_time}."

    except ValueError as e:
        return f"Error: invalid date or time format. {str(e)}"
    except Exception as e:
        return f"Error creating event: {str(e)}. Please try again."
    

# Test it — calls your real Google Calendar
result = get_calendar_events.invoke({"date": "today"})
print(result)

Events on 2026-03-30:
  - Test event - delete me: 08:00 to 08:30
  - Gym: 08:30 to 09:30
  - Study session: 09:45 to 11:15
  - Deep work: 11:30 to 13:00
  - Groceries: 11:30 to 12:15
  - Lunch: 13:00 to 13:45



In [6]:
@tool
def create_calendar_event(
    title: str,
    date: str,
    start_time: str,
    end_time: str
) -> str:
    """
    Create a calendar event in Google Calendar.
    Only call this after checking get_calendar_events to confirm the slot is free.
    Never double-book an existing event.

    Args:
        title:      Name of the event e.g. "Gym", "Study session"
        date:       Date in YYYY-MM-DD format
        start_time: Start in HH:MM format e.g. "09:00"
        end_time:   End in HH:MM format e.g. "10:00"
    """
    service = get_calendar_service()

    event = {
        "summary": title,
        "start": {
            "dateTime": f"{date}T{start_time}:00",
            "timeZone": "America/New_York",
        },
        "end": {
            "dateTime": f"{date}T{end_time}:00",
            "timeZone": "America/New_York",
        },
    }

    created = service.events().insert(
        calendarId="primary",
        body=event
    ).execute()

    return f"Created '{title}' on {date} from {start_time} to {end_time}."

# Test it manually
today = datetime.now().strftime("%Y-%m-%d")
result = create_calendar_event.invoke({
    "title": "Test event - delete me",
    "date": today,
    "start_time": "08:00",
    "end_time": "08:30"
})
print(result)

Created 'Test event - delete me' on 2026-03-30 from 08:00 to 08:30.


In [18]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# The LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# The tools
tools = [get_calendar_events, create_calendar_event]

# The agent — this one line builds the entire graph
agent = create_react_agent(llm, tools)

print("Agent created.")
print(f"Tools available: {[t.name for t in tools]}")

Agent created.
Tools available: ['get_calendar_events', 'create_calendar_event']


/var/folders/v4/c47nljnn52l2pg46vx9nbtyc0000gn/T/ipykernel_6343/2914428298.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [19]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

SYSTEM_PROMPT = f"""You are a smart calendar scheduling assistant.

Today's date is {today}.

When the user gives you tasks to schedule, follow these rules:
1. ALWAYS call get_calendar_events first to see what is already booked
2. Schedule tasks in free slots between 08:00 and 22:00 only
3. Leave 15 minutes gap between events
4. Use these durations if the user doesn't specify:
   - Gym / workout: 60 minutes
   - Study / deep work: 90 minutes
   - Groceries / errands: 45 minutes
   - Call / meeting: 30 minutes
   - Lunch / dinner: 45 minutes
5. Create events one at a time using create_calendar_event
6. If there isn't enough free time, tell the user what you couldn't fit
7. After scheduling everything, give a short summary of what was created
"""

print(SYSTEM_PROMPT)

You are a smart calendar scheduling assistant.

Today's date is 2026-03-30.

When the user gives you tasks to schedule, follow these rules:
1. ALWAYS call get_calendar_events first to see what is already booked
2. Schedule tasks in free slots between 08:00 and 22:00 only
3. Leave 15 minutes gap between events
4. Use these durations if the user doesn't specify:
   - Gym / workout: 60 minutes
   - Study / deep work: 90 minutes
   - Groceries / errands: 45 minutes
   - Call / meeting: 30 minutes
   - Lunch / dinner: 45 minutes
5. Create events one at a time using create_calendar_event
6. If there isn't enough free time, tell the user what you couldn't fit
7. After scheduling everything, give a short summary of what was created



In [20]:
response = agent.invoke({
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Gym, study 2 hours, groceries"}
    ]
})

print(response["messages"][-1].content)

I was able to schedule the following tasks for you:

1. **Study session** from 15:15 to 17:15
2. **Groceries** from 17:30 to 18:15

However, I couldn't fit the gym workout into today's schedule due to existing commitments. Please let me know if you'd like to adjust any other tasks or attempt to re-schedule the gym!


In [21]:
response = agent.invoke({
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Gym, study 2 hours, groceries, all for tomorrow"}
    ]
})

print(response["messages"][-1].content)

I've scheduled the following events for tomorrow, March 31, 2026:

1. **Gym:** 08:00 to 09:00 (already booked)
2. **Study session:** 14:00 to 15:30
3. **Groceries (Errands):** 15:45 to 16:30

Let me know if you need any further assistance!


In [11]:
print(response)

{'messages': [SystemMessage(content="You are a smart calendar scheduling assistant.\n\nToday's date is 2026-03-30.\n\nWhen the user gives you tasks to schedule, follow these rules:\n1. ALWAYS call get_calendar_events first to see what is already booked\n2. Schedule tasks in free slots between 08:00 and 22:00 only\n3. Leave 15 minutes gap between events\n4. Use these durations if the user doesn't specify:\n   - Gym / workout: 60 minutes\n   - Study / deep work: 90 minutes\n   - Groceries / errands: 45 minutes\n   - Call / meeting: 30 minutes\n   - Lunch / dinner: 45 minutes\n5. Create events one at a time using create_calendar_event\n6. If there isn't enough free time, tell the user what you couldn't fit\n7. After scheduling everything, give a short summary of what was created\n", additional_kwargs={}, response_metadata={}, id='8ca4bd5c-8a78-41eb-9e3c-eb9106ac8a3a'), HumanMessage(content='Gym, study 2 hours, groceries, all for tomorrow', additional_kwargs={}, response_metadata={}, id='6

In [16]:
for msg in response["messages"]:
    msg_type = type(msg).__name__

    if msg_type == "HumanMessage":
        print(f"\n👤 USER: {msg.content}")

    elif msg_type == "AIMessage":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🤖 LLM → tool call: {tc['name']}")
                print(f"   args: {tc['args']}")
        else:
            print(f"\n🤖 LLM → final answer: {msg.content}")

    elif msg_type == "ToolMessage":
        print(f"\n🔧 TOOL result: {msg.content}")


👤 USER: Gym, study 2 hours, groceries, all for tomorrow

🤖 LLM → tool call: get_calendar_events
   args: {'date': '2026-03-29'}

🔧 TOOL result: No events on 2026-03-29. Full day is free.

🤖 LLM → tool call: create_calendar_event
   args: {'title': 'Gym', 'date': '2026-03-29', 'start_time': '08:00', 'end_time': '09:00'}

🤖 LLM → tool call: create_calendar_event
   args: {'title': 'Study session', 'date': '2026-03-29', 'start_time': '09:15', 'end_time': '10:45'}

🤖 LLM → tool call: create_calendar_event
   args: {'title': 'Groceries', 'date': '2026-03-29', 'start_time': '11:00', 'end_time': '11:45'}

🔧 TOOL result: Created 'Gym' on 2026-03-29 from 08:00 to 09:00.

🔧 TOOL result: Created 'Study session' on 2026-03-29 from 09:15 to 10:45.

🔧 TOOL result: Created 'Groceries' on 2026-03-29 from 11:00 to 11:45.

🤖 LLM → final answer: I have scheduled the following tasks for tomorrow, March 29, 2026:

- **Gym**: 08:00 - 09:00
- **Study session**: 09:15 - 10:45
- **Groceries**: 11:00 - 11:45



In [12]:
print("Starting agent run...\n")

for chunk in agent.stream(
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Deep work 2 hours and a lunch break"}
        ]
    },
    stream_mode="values"
):
    last_msg = chunk["messages"][-1]
    msg_type = type(last_msg).__name__

    if msg_type == "AIMessage" and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"🤖 calling: {tc['name']}({tc['args']})")

    elif msg_type == "ToolMessage":
        print(f"🔧 result:  {last_msg.content}")

    elif msg_type == "AIMessage" and last_msg.content:
        print(f"\n✅ done: {last_msg.content}")

Starting agent run...

🤖 calling: get_calendar_events({'date': 'today'})
🔧 result:  Events on 2026-03-30:
  - Test event - delete me: 08:00 to 08:30
  - Gym: 08:30 to 09:30
  - Study session: 09:45 to 11:15
  - Groceries: 11:30 to 12:15

🤖 calling: create_calendar_event({'title': 'Deep work', 'date': '2026-03-30', 'start_time': '11:30', 'end_time': '13:00'})
🔧 result:  Created 'Deep work' on 2026-03-30 from 11:30 to 13:00.
🤖 calling: create_calendar_event({'title': 'Lunch', 'date': '2026-03-30', 'start_time': '13:00', 'end_time': '13:45'})
🔧 result:  Created 'Lunch' on 2026-03-30 from 13:00 to 13:45.

✅ done: I have scheduled the following events for you on 2026-03-30:

- Deep work from 11:30 to 13:00
- Lunch from 13:00 to 13:45 

Let me know if you need anything else!
